In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    Rescaling
)

# ==========================================
# 1. Load Dataset
# ==========================================

DATASET_PATH = "TrashType_Image_Dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_ds.class_names

print("\nClasses:")
print(class_names)

# Improve performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(
    buffer_size=AUTOTUNE
)

val_ds = val_ds.cache().prefetch(
    buffer_size=AUTOTUNE
)

# ==========================================
# 2. Build CNN Model
# ==========================================

model = Sequential([

    # Normalize pixel values (0-255 -> 0-1)
    Rescaling(1./255, input_shape=(224, 224, 3)),

    # Conv Block 1
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # Conv Block 2
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # Conv Block 3
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # Fully Connected
    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(64, activation='relu'),

    # Output Layer (6 classes)
    Dense(6, activation='softmax')
])

# ==========================================
# 3. Compile Model
# ==========================================

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show architecture
model.summary()

# ==========================================
# 4. Train Model
# ==========================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# ==========================================
# 5. Evaluate Model
# ==========================================

loss, accuracy = model.evaluate(val_ds)

print(f"\nValidation Accuracy: {accuracy*100:.2f}%")
print(f"Validation Loss: {loss:.4f}")

# ==========================================
# 6. Save Model
# ==========================================

model.save("trash_classifier.h5")

print("\nModel saved as trash_classifier.h5")

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ==========================================
# Load Model
# ==========================================
model = load_model("trash_classifier.h5")

# Nama kelas sesuai dataset
class_names = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash"
]

# ==========================================
# Load Image
# ==========================================
img_path = "/home/robot/Documents/sem6/machine_learning/CNN/glass/glass_018.jpg"

img = cv2.imread(img_path)

if img is None:
    raise FileNotFoundError(f"Gambar tidak ditemukan:\n{img_path}")

display_img = img.copy()

# ==========================================
# Preprocessing
# ==========================================
input_img = cv2.resize(img, (224, 224))
input_img = input_img.astype("float32") / 255.0
input_img = np.expand_dims(input_img, axis=0)

# ==========================================
# Prediction
# ==========================================
prediction = model.predict(input_img, verbose=0)

class_id = np.argmax(prediction)
confidence = float(np.max(prediction))

predicted_class = class_names[class_id]

print("Prediction :", predicted_class)
print("Confidence :", confidence * 100, "%")

# ==========================================
# Bounding Box (Manual)
# ==========================================

h, w = display_img.shape[:2]

x1 = int(w * 0.1)
y1 = int(h * 0.1)

x2 = int(w * 0.9)
y2 = int(h * 0.9)

cv2.rectangle(
    display_img,
    (x1, y1),
    (x2, y2),
    (0, 255, 0),
    3
)

label = f"{predicted_class} ({confidence*100:.1f}%)"

cv2.putText(
    display_img,
    label,
    (x1, y1 - 10),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (0, 255, 0),
    2
)

# ==========================================
# Show Image
# ==========================================
cv2.imshow("Trash Classification", display_img)

cv2.waitKey(0)
cv2.destroyAllWindows()